# ReplayHouse quick start
Experience replay on ClickHouse — running entirely in-process via chdb.
```
pip install replayhouse[embedded]
```

In [ ]:
import tempfile
import replayhouse

tmp = tempfile.mkdtemp()
store = replayhouse.connect(f"chdb://{tmp}/db")

In [ ]:
t = store.create(
    "experiences",
    columns={"task": "LowCardinality(String)", "steps": "JSON", "reward": "Float32"},
    capacity_rows=100_000, eviction="lowest_priority",
)

In [ ]:
import random
rng = random.Random(0)
t.insert([
    {"task": rng.choice(["web", "code"]), "steps": {"n": i}, "reward": rng.random(),
     "priority": 1.0}
    for i in range(1000)
])

In [ ]:
batch = t.sample(64, by="reward + 0.05", stratify_by="task")
len(batch), batch.rows[0]["task"]

In [ ]:
t.update_priorities(batch.ids, [0.5] * len(batch))
t.compact()
t.evict()

In [ ]:
batch.to_pandas().head()

In [ ]:
store.close()